In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
train = pd.read_csv("data/train.csv")

In [ ]:
test_texts = pd.read_csv("data/candidate_texts.csv")
test = pd.read_csv("data/test.csv")
test

,datapointID,coords
0,1,|0.904361|1.000347|0.875736|1.063815|0.806261|...
1,1,|0.864176|1.004364|0.816361|0.965073|0.745989|...
2,1,|0.812210|0.797805|0.716002|0.859149|0.763311|...
3,1,|0.935903|1.038246|0.873798|1.063595|0.808664|...
4,1,|0.815194|0.924060|0.756063|0.949227|0.747226|...
...,...,...
2616,50,|0.191120|0.944574|0.183643|0.965427|0.115751|...
2617,50,|-0.180338|0.842952|-0.213149|0.806235|-0.2442...
2618,50,|0.215246|0.857997|0.215392|0.786738|0.193160|...
2619,50,|0.068234|0.767058|0.110711|0.777773|0.047362|...


In [ ]:
from sklearn.preprocessing import StandardScaler
def compute_coords_per_batch_train(batch):
    coords = batch["coords"]
    text = batch["text"].iloc[0]
    # avem nevoie doar de un text pentru ca pe o secventa se repeta textul fiind aceleasi puncte
    coords = [parse_coords(i) for i in coords]

    coords = pd.DataFrame(coords).mean() # avem nevoie de centru
    #sc = StandardScaler()

   #  coords = sc.fit_transform([coords])

    return coords,text
def compute_coords_per_batch_test(batch):
    coords = batch["coords"]
    #text = batch["text"].iloc[0]
    # avem nevoie doar de un text pentru ca pe o secventa se repeta textul fiind aceleasi puncte
    coords = [parse_coords(i) for i in coords]

    coords = pd.DataFrame(coords).mean() # avem nevoie de centru
    #sc = StandardScaler()

   #  coords = sc.fit_transform([coords])

    return coords

In [ ]:
data_values = []
data_text = []
for i in range(300):
    df = train[train['id'] == i]
    data,text = compute_coords_per_batch_train(df)
    data_values.append(
         data
    )
    data_text.append(text)


In [ ]:
data_values = np.stack(data_values)

In [ ]:
from nltk.stem import PorterStemmer,WordNetLemmatizer
from nltk.tokenize import TweetTokenizer
import nltk

from nltk.corpus import stopwords,wordnet
def wordnet_tag(word):
    tag = nltk.pos_tag([word.lower()])[0][1][0].lower()

    tag_dict = {
        'n': wordnet.NOUN,
        'r' : wordnet.ADV,
        'v' : wordnet.VERB,
        'j': wordnet.ADJ,

    }

    return tag_dict.get(tag,wordnet.NOUN)
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
tokenizer = TweetTokenizer(preserve_case=False,reduce_len=True)
stop_words = stopwords.words('english')

def process_text(text):

    tokens = tokenizer.tokenize(text.lower())
    tokens = [token for token in tokens if token.isalpha() and token not in stop_words]

    tokens = [lemmatizer.lemmatize(word,wordnet_tag(word)) for word in tokens]

    return ' '.join(tokens)



In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.preprocessing import FunctionTransformer

In [ ]:
data_values

array([[ 0.9858473 , -0.00786468,  0.98824847, ..., -0.02481728,
         0.9885839 ,  0.00249215],
       [ 1.3811625 , -0.01183351,  1.378961  , ..., -0.03358839,
         1.384173  , -0.01412637],
       [ 0.10809135, -0.00521195,  0.11379597, ..., -1.1315286 ,
         0.28841045, -1.1023159 ],
       ...,
       [-1.0073785 , -0.00931638, -0.98095775, ..., -0.01707432,
        -1.0004779 , -0.03042579],
       [-0.9077971 , -0.7439893 , -0.8999796 , ..., -0.7415354 ,
        -0.905042  , -0.75128007],
       [-0.98236287, -0.9990394 , -0.9805237 , ..., -0.99337965,
        -0.9747417 , -0.99809265]], dtype=float32)

In [ ]:
data_text = pd.DataFrame(data_text)
data_text.columns = ["abc"]
data_text

,abc
0,"I used to pretend to believe in God. Mainly, I..."
1,She left a window open and the whole autumn ca...
2,There are mornings when I wake up and forget w...
3,He kept a list of all the things he had never ...
4,The last train left without anyone on it. And ...
...,...
295,"The summer I turned seventeen, everything smel..."
296,"She asked me to remember her as she was, not a..."
297,The mirror in the hallway showed a version of ...
298,We spoke in the dark because daylight made it ...


In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
preprocess = ColumnTransformer(
    [
        ("first",Pipeline(
            [
                ("func",FunctionTransformer(lambda x: x['abc'].apply(process_text))),
                ("tfidf",TfidfVectorizer(ngram_range=(1,3),analyzer="word",max_features=100000,max_df=0.98,min_df=1)),
            ]
        ),["abc"])
    ]
)

In [ ]:
data_text = preprocess.fit_transform(data_text)

In [ ]:
data_text

<300x8718 sparse matrix of type '<class 'numpy.float64'>'
	with 14540 stored elements in Compressed Sparse Row format>

In [ ]:
from sklearn.decomposition import TruncatedSVD

In [ ]:
svd_text = TruncatedSVD(n_components=300)
svd_vec = TruncatedSVD(n_components=300)

In [ ]:
train_text = svd_text.fit_transform(data_text)

In [ ]:
data_values

array([[ 0.9858473 , -0.00786468,  0.98824847, ..., -0.02481728,
         0.9885839 ,  0.00249215],
       [ 1.3811625 , -0.01183351,  1.378961  , ..., -0.03358839,
         1.384173  , -0.01412637],
       [ 0.10809135, -0.00521195,  0.11379597, ..., -1.1315286 ,
         0.28841045, -1.1023159 ],
       ...,
       [-1.0073785 , -0.00931638, -0.98095775, ..., -0.01707432,
        -1.0004779 , -0.03042579],
       [-0.9077971 , -0.7439893 , -0.8999796 , ..., -0.7415354 ,
        -0.905042  , -0.75128007],
       [-0.98236287, -0.9990394 , -0.9805237 , ..., -0.99337965,
        -0.9747417 , -0.99809265]], dtype=float32)

In [ ]:
train_vec = svd_vec.fit_transform(data_values)

In [ ]:
train_vec

array([[-3.5541172e+00, -1.8198761e+01,  1.0753766e+00, ...,
         1.3057422e-03, -8.5982215e-04, -6.3653216e-03],
       [-3.2080417e+00, -1.8304880e+01,  7.6393471e+00, ...,
         6.9626980e-04,  8.0876541e-04, -1.0139255e-03],
       [-5.3396553e-01,  6.8858707e-01, -1.9658451e+00, ...,
        -7.7263266e-04,  4.1463785e-04,  1.5308149e-03],
       ...,
       [-9.2779666e-01, -1.2724504e+00, -1.2025511e+01, ...,
        -2.6295586e-03, -1.6688034e-03, -1.6973978e-03],
       [ 2.8061302e+00, -8.0403173e-01, -1.3413010e+01, ...,
        -3.3281627e-05, -3.2434389e-03, -1.5637595e-03],
       [ 6.5444794e+00, -1.6255946e+00, -1.8357235e+01, ...,
         2.4185390e-03,  1.6222913e-03, -1.4994480e-03]], dtype=float32)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
from sklearn.preprocessing import Normalizer

In [ ]:
sc = Normalizer()

In [ ]:
ridge = Ridge()

In [ ]:
ridge

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [ ]:
len(train_vec)

300

In [ ]:
#train_vec = sc.fit_transform(train_vec)

In [ ]:
ridge.fit(train_vec,train_text)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [ ]:
ridge.get_params()

{'alpha': 1.0,
 'copy_X': True,
 'fit_intercept': True,
 'max_iter': None,
 'positive': False,
 'random_state': None,
 'solver': 'auto',
 'tol': 0.0001}

In [ ]:
ridge.score(train_vec,train_text)

0.12352485439736886

In [ ]:
test_texts2 = test_texts['text']

In [ ]:
test_texts2 = pd.DataFrame(test_texts2)

In [ ]:
test_texts2.columns = ["abc"]

In [ ]:
test_texts2 = preprocess.transform(test_texts2)

In [ ]:
test_texts2

<500x8718 sparse matrix of type '<class 'numpy.float64'>'
	with 19097 stored elements in Compressed Sparse Row format>

In [ ]:
test_texts2 = svd_text.transform(test_texts2)

In [ ]:
datatest_values = []
for i in range(1,51):
    df = test[test['datapointID'] == i]
    data = compute_coords_per_batch_test(df)
    datatest_values.append(
         data
    )


In [ ]:
datatest_values = np.stack(datatest_values)

In [ ]:
len(datatest_values)

50

In [ ]:
datatest_values = svd_vec.transform(datatest_values)

In [ ]:
predicts = []
for i in datatest_values:
    best_score = -10000
    best_id = -10000
    #print(i.shape)
    mapped_vec = ridge.predict([i])
    mapped_vec = mapped_vec
    for j in range(500):
        if j not in predicts:
            euclid_sim   = 1 / (1 + np.linalg.norm(mapped_vec - test_texts2[j]))
            if euclid_sim  > best_score:
                best_score = euclid_sim
                best_id = j
    predicts.append(best_id)


0.6554998143022598
0.6667659121572008
0.6523936327528207
0.6508457733605943
0.6558853431224972
0.6528805695341288
0.664311431813619
0.6521156357025683
0.6558951595465703
0.6514699396198111
0.6491528858201225
0.6539935548206195
0.6510242162962668
0.6444415291504865
0.6490318378601827
0.6442933934596627
0.63547223622558
0.6496239117647864
0.6396160243489135
0.6352804991655127
0.6522174409239759
0.632086739044549
0.6372828903124431
0.6309998329744922
0.632360739326179
0.6355238148051812
0.6505713313432804
0.6308902272724994
0.6335395386654502
0.6322087897594802
0.6412996197849284
0.6321257288355483
0.6320448679472381
0.6274172475728055
0.6179327534679379
0.6424349253040436
0.6272286504134069
0.6340047222016897
0.6259513546575669
0.6184388343742799
0.6331564268616842
0.633400706505503
0.6324857664011639
0.6382943906519807
0.6306848843954456
0.6320727646253987
0.6330296332053994
0.6273563233113945
0.6175304280721284
0.6257519626292615


In [ ]:
#aici creez submissionul
subm= []
for i in range(50):
    subm.append(
        {
            "subtaskID":1,
            "datapointID": i + 1,
            'answer':predicts[i]

        }
    )
subm = pd.DataFrame(subm)
subm.to_csv("submission.csv",index=False)